In [1]:
!pip install torch==2.9.0 torchaudio==2.9.0 torchcodec==0.8.0 datasets==4.4.1 transformers==4.57.3 evaluate==0.4.6 accelerate==1.12 jiwer tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 140.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import torch
import numpy as np
import re
import gc
import unicodedata
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_from_disk, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from transformers.trainer_utils import get_last_checkpoint

DATASET_PATH = "/content/drive/MyDrive/Processed_Hindi_Parquet_Dataset"
OUTPUT_DIR   = "/content/drive/MyDrive/whisper-small-lora-ap-3"

MODEL_ID = "openai/whisper-small"
LANGUAGE = "hindi"
TASK = "transcribe"
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100
EVAL_STEPS = 400
SAVE_STEPS = 400
MAX_STEPS = 12000

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU (⚠️ Slow)")


metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# preprocessing
def prepare_dataset(batch):
    batch["text"] = normalize_hindi(batch["text"])
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

# metrics
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "wer": 100 * metric_wer.compute(predictions=pred_str, references=label_str),
        "cer": 100 * metric_cer.compute(predictions=pred_str, references=label_str),
    }


def normalize_hindi(text):

    # 1. Unicode normalize (important)
    text = unicodedata.normalize("NFC", text)

    # 2. Lowercase English if present
    text = text.lower()

    # 3. Remove excessive punctuation (keep . and ? if you want)
    text = re.sub(r"[“”‘’\"\'\-~!@#$%^&*()_+=:;<>/\\|`\[\]{}…]", " ", text)

    # 4. Early whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    # 5. Normalize nukta chars (harmless + reduces noise)
    nukta_map = {
        "क़": "क", "ख़": "ख", "ग़": "ग", "ज़": "ज",
        "ड़": "ड", "ढ़": "ढ", "फ़": "फ", "य़": "य",
    }
    for k, v in nukta_map.items():
        text = text.replace(k, v)

    # 6. DO NOT touch chandrabindu/anusvara (keep nasalization)
    #    --> no changes here

    # 7. Normalize "हैं" → "है" (huge consistency win)
    text = text.replace("हैं", "है")

    # 9. Remove English letters
    text = re.sub(r"[a-zA-Z]", "", text)

    # 10. Final space collapse
    text = re.sub(r"\s+", " ", text).strip()

    return text

# main
if __name__ == "__main__":

    print("Loading processor and model...")
    processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    model.to(device)

    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = None

    model.generation_config.no_repeat_ngram_size = 3
    model.generation_config.repetition_penalty = 1.25
    model.generation_config.temperature = 0.7
    model.generation_config.top_p = 0.9

    # lora config
    print("Applying LoRA...")
    peft_config = LoraConfig(
        inference_mode=False,
        r=32,
        lora_alpha=8,
        target_modules=["q_proj", "k_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    print("Loading dataset...")
    ds = load_from_disk(DATASET_PATH)

    if "validation" not in ds:
        ds_splits = ds.train_test_split(test_size=50, seed=42)
        ds = DatasetDict({"train": ds_splits["train"], "validation": ds_splits["test"]})
    if os.path.exists("/content/drive/MyDrive/Processed_Hindi_Mapped_Final"):
        ds_prepared = load_from_disk("/content/drive/MyDrive/Processed_Hindi_Mapped_Final")
    else:
        ds_prepared = ds.map(
            prepare_dataset,
            remove_columns=ds["train"].column_names,
            num_proc=8,
        )
        ds_prepared.save_to_disk("/content/drive/MyDrive/Processed_Hindi_Mapped_Final")

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

    # training args
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        max_steps = MAX_STEPS,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,

        fp16=True,

        predict_with_generate=True,
        generation_max_length=225,

        save_steps=SAVE_STEPS,
        save_total_limit=20,

        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,

        remove_unused_columns=False,
        label_names=["labels"],
        max_grad_norm=1.0,
        report_to = "none"
    )

    # trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=ds_prepared["train"],
        eval_dataset=ds_prepared["validation"],
        data_collator=data_collator,
        tokenizer=processor.feature_extractor,
        compute_metrics=compute_metrics,
    )

    #checkpoint_dir = get_last_checkpoint(OUTPUT_DIR)
    #print("Resuming from:", checkpoint_dir)

    # train
    print("Starting training...")
    trainer.train()

    model.save_pretrained(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)

    print("Training Done!")

Using CUDA GPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading processor and model...
Applying LoRA...
trainable params: 5,308,416 || all params: 247,043,328 || trainable%: 2.1488
Loading dataset...


/tmp/ipython-input-3816856138.py:222: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting training...


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
